# Đánh giá toàn diện bộ dữ liệu PROVEDIt 1-5 Person (Filtered)

Notebook này cung cấp các phân tích và thống kê chi tiết về bộ dữ liệu PROVEDIt, bao gồm:
1. Thống kê quy mô tập dữ liệu (số lượng file, số dòng).
2. Cấu trúc và Schema của dữ liệu.
3. Phân phối các giá trị đặc biệt (Nhiễm sắc thể X, Y ở AMEL và tín hiệu OL).
4. Đánh giá chất lượng tín hiệu (Peak Height/RFU).

*Lưu ý: Do kích thước dữ liệu rất lớn, notebook này sẽ tiến hành đọc mẫu một số file đại diện và tổng hợp thống kê theo từng thư mục thí nghiệm.*

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình thư mục gốc
base_dir = '/Users/nguyenthithutam/Desktop/TAWSEEM/PROVEDIt_1-5-Person CSVs Filtered'

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

## 1. Thống kê cấu trúc thư mục và số lượng file
Đếm số lượng file CSV theo từng bộ Kit, phân bố số lượng người (1-5 Person) và thời gian tiêm (Injection Time).

In [ ]:
stats = {
    'kits': {},
    'persons': {},
    'injection_times': {},
    'total_files': 0
}

for kit_folder in sorted(os.listdir(base_dir)):
    kit_path = os.path.join(base_dir, kit_folder)
    if not os.path.isdir(kit_path): continue
    
    stats['kits'][kit_folder] = 0
    
    for root, dirs, files in os.walk(kit_path):
        for file in files:
            if file.endswith('.csv'):
                stats['total_files'] += 1
                stats['kits'][kit_folder] += 1
                
                rel_path = os.path.relpath(root, kit_path)
                parts = rel_path.split(os.sep)
                if len(parts) >= 2:
                    person_dir, sec_dir = parts[0], parts[1]
                    stats['persons'][person_dir] = stats['persons'].get(person_dir, 0) + 1
                    stats['injection_times'][sec_dir] = stats['injection_times'].get(sec_dir, 0) + 1

print(f"Tổng số tệp CSV: {stats['total_files']}\n")
print("Phân phối theo Kit:")
for k, v in stats['kits'].items(): print(f"  - {k}: {v} tệp")

print("\nPhân phối theo số lượng người (NOC):")
for k, v in sorted(stats['persons'].items()): print(f"  - {k}: {v} tệp")

print("\nPhân phối theo thời gian tiêm (Injection Time):")
for k, v in sorted(stats['injection_times'].items()): print(f"  - {k}: {v} tệp")

## 2. Kích thước tập dữ liệu và Schema
Lấy mẫu ngẫu nhiên vài file để xem các cột hiện có, các kiểu dữ liệu và tổng số dòng xấp xỉ.

In [ ]:
sample_file = os.path.join(base_dir, 'PROVEDIt_1-5-Person CSVs Filtered_3130_IDPlus28cycles', '1-Person', '5 sec', 'RD14-0003_IP_5sec_GM_F_1P.csv')
df_sample = pd.read_csv(sample_file, low_memory=False)

print(f"Kích thước file mẫu: {df_sample.shape[0]} dòng, {df_sample.shape[1]} cột")
print("\nCác cột meta chính:")
print(df_sample.columns[:3].tolist())

print("\nCác cột dữ liệu (Allele, Size, Height):")
print(df_sample.columns[3:12].tolist(), "...")

display(df_sample.head())

## 3. Đánh giá Marker và Dye (Locus huỳnh quang)
Phân tích xem các Marker (Locus) nào được sử dụng và số lượng dòng dữ liệu đại diện cho từng loại.

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df_sample, x='Marker', order=df_sample['Marker'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title('Tần suất dòng dữ liệu theo Marker (Mẫu 1-Person, 5 giây)')
plt.ylabel('Số lượng dòng')
plt.tight_layout()
plt.show()

print("Số lượng dòng theo Dye (Kênh màu học):")
print(df_sample['Dye'].value_counts())

## 4. Phân tích Dữ liệu Điển hình: OL và X/Y (Nhiễm sắc thể Giới tính)
Dữ liệu DNA thô thường chứa các tín hiệu `OL` (Off-Ladder: allele không khớp kích thước chuẩn) và `X`, `Y` tại locus AMEL.
Dưới đây là đoạn code đếm tổng quan các giá trị này.

In [ ]:
def detect_anomalies(df):
    allele_cols = [c for c in df.columns if c.startswith('Allele')]
    
    # Đếm OL
    ol_count = (df[allele_cols] == 'OL').sum().sum()
    
    # Đếm X, Y
    x_count = (df[allele_cols] == 'X').sum().sum()
    y_count = (df[allele_cols] == 'Y').sum().sum()
    
    # Tín hiệu AMEL
    amel_rows = df[df['Marker'] == 'AMEL'].shape[0]
    
    print(f"Tổng số giá trị 'OL': {ol_count}")
    print(f"Tổng số giá trị 'X': {x_count}")
    print(f"Tổng số giá trị 'Y': {y_count}")
    print(f"Tổng số dòng thuộc Marker AMEL: {amel_rows}")
    
    return df[df['Marker'] == 'AMEL'].head()

detect_anomalies(df_sample)

## 5. Phân tích Phân phối Tín hiệu Đỉnh (Peak Heights / RFU)
Relative Fluorescence Units (RFU) biểu thị cấu hình tín hiệu của các Allele. Việc hiểu phân phối chiều cao giúp quyết định Threshold để lọc nhiễu (stutters).

In [ ]:
# Trích xuất tất cả các giá trị Height hợp lệ trong 3 cột đầu
height_cols = ['Height 1', 'Height 2', 'Height 3']
heights = pd.to_numeric(df_sample[height_cols].values.flatten(), errors='coerce')
heights = heights[~np.isnan(heights)]  # Loại bỏ NaN

print("Thống kê mô tả Height (RFU):")
print(pd.Series(heights).describe())

# Vẽ histogram log-scale
plt.figure(figsize=(10, 5))
sns.histplot(heights, bins=50, log_scale=(True, False))
plt.title('Phân phối Chiều cao Đỉnh - Log Scale (Mẫu 1-Person 5s)')
plt.xlabel('Log(Height)')
plt.ylabel('Tần suất')
plt.axvline(np.median(heights), color='red', linestyle='--', label=f'Median: {np.median(heights):.0f}')
plt.legend()
plt.show()

## 6. Tổng kết và Đề xuất Tiền xử lý (Preprocessing Recommendations)

Dựa trên các đánh giá trên, khi xử lý dữ liệu cho mô hình Pipeline (ví dụ: mô hình TAWSEEM CNN / XGBoost), ta cần thực hiện các quy trình:
1. **Xử lý AMEL (Giới tính):** Các cột Allele chứa chuỗi `'X'` và `'Y'`. Cần loại bỏ toàn bộ các dòng `Marker == 'AMEL'` vì giới tính không đóng góp cho bài toán xác định Số người (NOC).
2. **Xử lý OL (Off-Ladder):** Đánh dấu các chuỗi `'OL'` bằng một feature nhị phân riêng biệt (ví dụ `OL_ind = 1`) và thay thế giá trị chuỗi thành một hằng số thực (như `0.0` hoặc `-1.0`) để giữ tính liên tục của tensor numeric.
3. **Lọc nhiễu:** RFU (Height) có phương sai rất lớn. Có thể cần chuẩn hóa (Normalization) hoặc lọc bỏ các tín hiệu nhỏ (dưới chuẩn Threshold của PTN).
4. **Missing Values:** Dữ liệu có rất nhiều cột (lên tới Allele 100), đa số sẽ là `NaN`. Có thể cần giới hạn lại số lượng Allele tối đa (ví dụ cắt ở Allele 10) để bảo toàn kích thước feature matrix hợp lý.
